<a href="https://colab.research.google.com/github/Kaviyarasi-Sasiperumal/AI_-Based_-Document-_Search_-and-_Knowledge-_Retrieval_-with-_Conversational_Interface/blob/main/Milestone_4_Deployment_%26__Final_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf sentence-transformers faiss-cpu transformers gradio torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.7 MB/s eta 0:00:00


In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
import gradio as gr
import time


In [ ]:
documents = []
metadata = []
embedder = SentenceTransformer("all-MiniLM-L6-v2")
index = None
llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausalLM', 'FlexOlmoF

In [ ]:
def chunk_text(text, chunk_size=250, overlap=100):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i+chunk_size])
    return chunks


def load_document(file):
    global documents, metadata, index

    documents = []
    metadata = []

    reader = PdfReader(file.name)

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            chunks = chunk_text(text)
            for chunk in chunks:
                documents.append(chunk)
                metadata.append(f"{file.name} - page {page_num+1}")

    if not documents:
        return "❌ No readable text found in document."

    # Create embeddings & FAISS index
    embeddings = embedder.encode(documents)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings))

    return f"✅ Document loaded successfully! Pages indexed: {len(metadata)}"


In [ ]:
!pip install gradio PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.2 MB/s eta 0:00:00


In [ ]:
def chat_function(user_input, history):
    start_time = time.perf_counter()

    if user_input.lower() in ["hi", "hello", "hey"]:
        return "👋Hello! Ask Questions from Document."

    if not document_text:
        return "Please upload a document first."

    answer = get_answer_from_document(user_input)

    response_time = (time.perf_counter() - start_time) * 1000
    response_time = round(response_time, 2)

    return f"""📄 Answer:
{answer}

⏱ Response Time: {response_time} ms
"""


# -------- UI --------
with gr.Blocks() as demo:
    gr.Markdown("## 🤖  Document Chatbot ")

    with gr.Row():
        # ---- LEFT SIDE ----
        with gr.Column(scale=1):
            gr.Markdown("### 📂 Upload Document")
            file_input = gr.File(label="Upload PDF / TXT")
            status = gr.Textbox(label="Status", interactive=False)
            stats_box = gr.Textbox(label="Document Statistics", interactive=False)

            file_input.change(
                load_document,
                file_input,
                outputs=[status, stats_box]
            )

        # ---- RIGHT SIDE ----
        with gr.Column(scale=3):
            gr.ChatInterface(
                fn=chat_function,
                title="Ask questions"
            )

demo.launch()


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fc065d47b2b21f3af2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
